# Crypto Candlestick-CNN — training + scientific harness

**Standalone.** Reads only `crypto_market_data_2018_2026.parquet` (from `crypto_data_prep.ipynb`).
Writes only `crypto_*` artifacts. It cannot touch the equities pipeline: every path is namespaced
(`crypto_image_store/`, `crypto_preds/`, `crypto_results.csv`), so nothing can collide with
`image_store/`, `preds/` or `results.csv`.

The deliverable is an **honest, corrected result**, not a winning Sharpe.

---

## PRE-REGISTERED CONFIG GRID — frozen before any run

| | |
|---|---|
| **Image sizes** | **{5, 20}** |
| **Horizons** | **{5, 20}** |
| **Total configs** | **4** = the entire family of tests |
| **BAGGING** | **False** (paper-faithful: one fixed split, diversity from random init) |
| **Ensemble** | 5 members per config |
| **Cost grid** | **0 / 10 / 25 / 50 bps per side** |
| **Annualization** | **365** — everywhere |

**No 60-day images. No 60-day horizons.** If those are wanted later they are a **separate
pre-registered family** and get their own correction — they are never folded back into this one.
Widening a family after seeing results is how a null becomes a "finding".

**This grid IS the search.** The scientific question is not "is the best config good?" — that config
was *selected* for being the best, which is circular. It is: **does any config survive
multiple-testing correction across all 4?** Bonferroni multiplier is therefore **4**, fixed.

---

## The 24/7 correctness fixes (why this is not just the equities notebook with a new parquet)

Equities math assumes 252 trading days and a calendar with weekends. Crypto has neither, and the
mismatch is not cosmetic — it silently corrupts the Sharpe.

**1. Annualization is 365, everywhere.** `ppy = 365/H`, and `sqrt(365/H)` in the Sharpe *and* in the
block-bootstrap. Using 252 on 24/7 data understates annualized Sharpe by `sqrt(365/252) = 1.20` —
a 20% error applied uniformly, which would flatter or flatten every number in the study.
Verified upstream: BTCUSDT has a bar on **100.00%** of calendar days (3,257/3,257).

**2. The rebalance calendar is built from the continuous date grid**, not by slicing whatever dates
happen to appear. The equities code used `dates[::H]` on observed dates. That is correct only if
every step is one bar; here it is built with `pd.date_range(..., freq=f'{H}D')`, so rebalances are
**exactly H calendar days apart** by construction, and cannot drift if a date is thin.

**3. The holding period is enforced to be EXACTLY H calendar days.** `end_idx + H` is a *positional*
offset. Within an era, bars are ~daily but a gap of up to 7 days can exist, so a positional offset
can quietly produce a 7-day "5-day" return. Any sample whose realized span is not exactly `H` days
is **dropped** (§4). This makes `365/H` the true periods-per-year rather than an approximation.

## Carried over unchanged — the guards that matter

- **Blank-image mask fix** (per-day NaN flag *then* roll, so incomplete leading windows stay `NaN`)
  plus `DROP_INCOMPLETE=True` at load.
- `SIGNAL_GATE=False` — the CNN's sample set depends on the **image alone** (paper-faithful).
- Samples are gated to `InUniverse` dates; lookback prices from non-universe days are still real
  prices and remain usable.
- Non-overlapping decile portfolios; the real Han–Zhou–Zhu TREND fitted at scoring time.
- Bonferroni + BH, block-bootstrap Sharpe CI, cost grid, survivorship decomposition.

---

## Train/test split — pre-registered, stated before running

| | |
|---|---|
| **Train/val** | 2018-01-01 → **2020-12-31** |
| **Out-of-sample** | **2021-01-01** → 2026-06-30 (~5.5 years) |

Rationale, fixed in advance: the cross-rate approximation from Stage 0/1 is concentrated almost
entirely in the training years (**86.6%** of 2018 rows are BTC-cross-rated, 59.6% in 2019, 21.9% in
2020, but **≈0.6% from 2021 onward**). So the **test window is essentially pure USDT prints** and the
approximation cannot contaminate the result being reported. OOS gives ~401 non-overlapping periods
at H=5 and ~100 at H=20.

## 0. Imports, device, and the locked configuration

In [24]:
import os, gc, sys, time, json, tempfile, threading, platform
import multiprocessing as mp
try: mp.set_start_method('fork', force=True)     # fork lets render workers inherit data copy-on-write
except RuntimeError: pass
import concurrent.futures as cf
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

try:
    import psutil; _PROC = psutil.Process()
except ImportError:
    _PROC = None

# ---------------------------------------------------------------- device
if torch.cuda.is_available():
    BACKEND = 'cuda'
    DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())]
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    BACKEND, DEVICES = 'mps', ['mps']
else:
    BACKEND, DEVICES = 'cpu', ['cpu']
CPU = mp.cpu_count()

# ============================================================================
# LOCKED / PRE-REGISTERED  —  do not tune, do not expand
# ============================================================================
IMAGE_SIZES = [5, 20]        # LOCKED: no 60
HORIZONS    = [5, 20]        # LOCKED: no 60
BAGGING     = False          # LOCKED
ANNUALIZER  = 365.0          # LOCKED: crypto is 24/7 (measured: 100.0% of calendar days)
COST_GRID   = [0, 10, 25, 50]  # LOCKED: bps per side
N_CONFIGS   = len(IMAGE_SIZES) * len(HORIZONS)      # 4 -> the Bonferroni multiplier

TRAIN_END  = '2020-12-31'    # pre-registered
OOS_START  = '2021-01-01'
OOS_END    = '2026-06-30'

# ---------------------------------------------------------------- paper knobs
BATCH_SIZE = 128
EPOCHS     = 100
LR         = 1e-5
PATIENCE   = 5
USE_AMP    = False           # False = fp32 (paper-exact)
ENSEMBLE_SIZE     = 5
TRAIN_FRAC        = 0.7
SHARED_SPLIT_SEED = 42
SIGNAL_GATE       = False    # CNN samples gated by the IMAGE only (paper-faithful)
DROP_INCOMPLETE   = True     # drop windows without a full lookback (blank-image bug guard)
TESTING           = False    # True -> 3 symbols, 1 config, 2 members

MODELS_PER_GPU = 2
if BACKEND == 'cuda':
    MAX_PARALLEL = max(1, len(DEVICES) * MODELS_PER_GPU)
    NUM_WORKERS  = max(0, min(2, CPU // MAX_PARALLEL))
else:
    MAX_PARALLEL = 1
    NUM_WORKERS  = min(4, max(0, CPU - 2))

# ---------------------------------------------------------------- paths (crypto-namespaced)
PARQUET_PATH = './crypto_market_data_2018_2026.parquet'
IMG_DIR      = './crypto_image_store'      # NOT ./image_store  (that is the equities store)
OUT_DIR      = '.'
PRED_DIR     = os.path.join(OUT_DIR, 'crypto_preds')     # NOT ./preds
RESULTS_CSV  = os.path.join(OUT_DIR, 'crypto_results.csv')  # NOT ./results.csv
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True)

# ---------------------------------------------------------------- logging
_LOCK = threading.Lock(); _T0 = time.time()
def log(msg, tag=''):
    el = time.time() - _T0
    with _LOCK:
        print(f"[{el//3600:02.0f}:{(el%3600)//60:02.0f}:{el%60:04.1f}]{(' '+tag) if tag else ''} {msg}", flush=True)

def mem_str(dev=None):
    parts = []
    if _PROC is not None: parts.append(f"rss {_PROC.memory_info().rss/1e9:.1f}G")
    if BACKEND == 'cuda' and dev is not None:
        i = dev.index or 0
        parts.append(f"gpu{i} {torch.cuda.memory_allocated(i)/1e9:.1f}/{torch.cuda.memory_reserved(i)/1e9:.1f}G")
    elif BACKEND == 'mps':
        try: parts.append(f"mps {torch.mps.current_allocated_memory()/1e9:.1f}G")
        except Exception: pass
    return " ".join(parts)

def free_memory(dev=None):
    gc.collect()
    if BACKEND == 'cuda':
        if dev is not None and dev.type == 'cuda':
            with torch.cuda.device(dev): torch.cuda.empty_cache()
        else:
            for i in range(len(DEVICES)):
                with torch.cuda.device(i): torch.cuda.empty_cache()
    elif BACKEND == 'mps':
        try: torch.mps.empty_cache()
        except Exception: pass

log(f"host={platform.node()} backend={BACKEND} devices={DEVICES} vCPU={CPU}")
log(f"LOCKED grid: images {IMAGE_SIZES} x horizons {HORIZONS} = {N_CONFIGS} configs | BAGGING={BAGGING}")
log(f"LOCKED annualization={ANNUALIZER:.0f} | cost grid {COST_GRID} bps/side")
log(f"split: train<={TRAIN_END} | OOS {OOS_START}..{OOS_END}")
log(f"paths: img={IMG_DIR} preds={PRED_DIR} results={RESULTS_CSV}")

[00:00:00.0] host=Yashs-MacBook-Pro.local backend=mps devices=['mps'] vCPU=10
[00:00:00.0] LOCKED grid: images [5, 20] x horizons [5, 20] = 4 configs | BAGGING=False
[00:00:00.0] LOCKED annualization=365 | cost grid [0, 10, 25, 50] bps/side
[00:00:00.0] split: train<=2020-12-31 | OOS 2021-01-01..2026-06-30
[00:00:00.0] paths: img=./crypto_image_store preds=./crypto_preds results=./crypto_results.csv


## 1. Market data

The parquet carries `Symbol` (not `Ticker`) and `InUniverse` (not `InIndex`). Assets split by the
ticker-reuse / redenomination guard carry a `#n` era suffix and are **independent assets** — which is
correct, because they are.

In [25]:
if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(f"parquet not found: {PARQUET_PATH} -- run crypto_data_prep.ipynb first")
df_market = pd.read_parquet(PARQUET_PATH)
df_market['Date'] = pd.to_datetime(df_market['Date'])

for c in ['Date','Symbol','Open','High','Low','Close','Volume','InUniverse','VolRegime','Source']:
    if c not in df_market.columns:
        raise RuntimeError(f"parquet missing required column {c!r} -- regenerate with crypto_data_prep.ipynb")

if TESTING:
    keep = df_market.loc[df_market.InUniverse, 'Symbol'].unique()[:3]
    df_market = df_market[df_market['Symbol'].isin(keep)].copy()
    IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5], [5], 2
    log("TESTING MODE: 3 symbols, 1 config, 2 members")

MA_LAGS     = [3, 5, 10, 20, 50, 100, 200]
MA_COLS     = [f'MA{L}' for L in MA_LAGS]
IMAGE_COLS  = ['Open','High','Low','Close','Volume']   # the CNN sees ONLY these
SIGNAL_COLS = ['MOM','STR','WSTR'] + MA_COLS           # benchmarks, NOT CNN inputs
META_COLS   = SIGNAL_COLS

log(f"parquet: {len(df_market):,} rows | {df_market['Symbol'].nunique():,} assets | "
    f"{df_market['Date'].min().date()} -> {df_market['Date'].max().date()}")
log(f"universe rows: {int(df_market.InUniverse.sum()):,} "
    f"({100*df_market.InUniverse.mean():.1f}%) | mem={mem_str()}")

# ---- 24/7 sanity: the universe date grid must be continuous, else the H-day rebalance drifts ----
udates = pd.DatetimeIndex(np.sort(df_market.loc[df_market.InUniverse, 'Date'].unique()))
gaps = udates.to_series().diff().dt.days.dropna()
log(f"universe date grid: {len(udates):,} dates, max gap {gaps.max():.0f} day(s) "
    f"-> {'CONTINUOUS (24/7 confirmed)' if gaps.max() <= 1 else 'HAS GAPS -- investigate'}")
names_per_date = df_market[df_market.InUniverse].groupby('Date').size()
log(f"names per date: min {names_per_date.min()} max {names_per_date.max()} "
    f"mean {names_per_date.mean():.1f}")

# ---- OOS cross-rate purity: the approximation must live in TRAIN, not in the reported window ----
oos = df_market[(df_market.Date >= OOS_START) & df_market.InUniverse]
log(f"OOS rows sourced from the BTC cross-rate: {100*(oos.Source=='BTCX').mean():.2f}% "
    f"(train-era contamination stays out of the reported result)")

[00:00:00.8] parquet: 887,326 rows | 662 assets | 2018-01-01 -> 2026-06-30
[00:00:00.8] universe rows: 309,276 (34.9%) | mem=rss 0.6G mps 0.1G
[00:00:00.8] universe date grid: 3,093 dates, max gap 1 day(s) -> CONTINUOUS (24/7 confirmed)
[00:00:00.8] names per date: min 97 max 100 mean 100.0
[00:00:00.8] OOS rows sourced from the BTC cross-rate: 0.20% (train-era contamination stays out of the reported result)


## 2. Image generation — exact PIL rendering (unchanged from the equities notebook)

In [26]:
def _height(n_days): return 32 if n_days == 5 else (64 if n_days == 20 else 96)
def img_shape(n_days): return _height(n_days), n_days * 3

def generate_image_uint8(window_df, n_days):
    height, width = img_shape(n_days)
    img = Image.new('L', (width, height), color=0)
    if len(window_df) == 0:
        return np.zeros((height, width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close == 0 or np.isnan(first_close):
        return np.zeros((height, width), dtype=np.uint8)
    draw = ImageDraw.Draw(img)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p == min_p: max_p = min_p + 1e-5
    if max_v == 0:     max_v = 1e-5
    price_h = int(height * 0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1), 0, price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1), price_h, height-1))
    ma_pts = []
    for i in range(len(window_df)):
        x = i*3 + 1
        oy, hy, ly, cy = p2y(prices[i,0]), p2y(prices[i,1]), p2y(prices[i,2]), p2y(prices[i,3])
        draw.line([(x,hy), (x,ly)], fill=255, width=1)
        draw.line([(x-1,oy), (x,oy)], fill=255, width=1)
        draw.line([(x,cy), (x+1,cy)], fill=255, width=1)
        draw.line([(x,height-1), (x,v2y(volumes[i]))], fill=255, width=1)
        ma_pts.append((x, p2y(ma[i])))
    if len(ma_pts) > 1:
        draw.line(ma_pts, fill=255, width=1)
    return np.array(img, dtype=np.uint8)

## 3. Image store — the blank-image mask fix, kept

The bug this guards against: reducing NaN with `rolling(n).sum().sum(axis=1) == 0` turns an
all-`NaN` leading row into `0.0`, which **passes**, then `iloc[idx-n+1 : idx+1]` slices with a
negative start → an **empty window** → an all-black image paired with a real label.

The fix reduces NaN to a per-day flag **first**, then rolls: an incomplete leading window stays
`NaN`, and `NaN == 0` is `False`, so it is correctly excluded.

`DROP_INCOMPLETE=True` also filters `end_idx < n_days-1` at load — a no-op on a clean store, and a
rescue for a stale one.

In [27]:
def align_store(m, df, verbose=False):
    '''Boolean mask of store rows whose image still matches the current parquet.'''
    pos_df = (df.sort_values(['Symbol', 'Date'])
                .assign(pos=lambda x: x.groupby('Symbol').cumcount(),
                        end_date=lambda x: x['Date'].dt.strftime('%Y-%m-%d'))
                [['Symbol', 'end_date', 'pos']]
                .rename(columns={'Symbol': 'symbol'}))
    mm = m.merge(pos_df, on=['symbol', 'end_date'], how='left')
    ok = mm['pos'].notna() & (mm['pos'].values == mm['end_idx'].values)
    return ok.values, mm

def _index_symbol(args):
    symbol, s_df, n_days, gate = args
    s_df = s_df.sort_values('Date').reset_index(drop=True)
    cols = IMAGE_COLS + (SIGNAL_COLS if gate else [])
    bad_day    = s_df[cols].isna().any(axis=1).astype(float)
    window_bad = bad_day.rolling(window=n_days).sum()          # NaN on the first n_days-1 rows
    valid      = np.where(window_bad.values == 0)[0]           # -> leading rows correctly excluded
    if 'InUniverse' in s_df.columns:                           # sample ONLY on universe dates;
        valid = valid[s_df['InUniverse'].values[valid].astype(bool)]   # lookback prices from
                                                                        # non-universe days are real
                                                                        # prices and remain usable
    sig = {c: s_df[c].values for c in META_COLS}
    recs = [dict({'symbol': symbol, 'end_idx': int(i), 'end_date': str(s_df['Date'].iloc[i].date())},
                 **{c: float(sig[c][i]) for c in META_COLS})
            for i in valid]
    return symbol, s_df, recs

_G = {}
def _render_chunk(rng):
    lo, hi = rng
    mm = np.memmap(_G['dat'], dtype=np.uint8, mode='r+', shape=_G['shape'])
    nd = _G['n_days']
    for i in range(lo, hi):
        s = _G['recs'][i]; sdf = _G['grouped'][s['symbol']]; e = s['end_idx']
        mm[i, 0] = generate_image_uint8(sdf.iloc[e-nd+1 : e+1], nd)
    mm.flush(); del mm
    return hi - lo

def store_paths(n_days):
    return (os.path.join(IMG_DIR, f'raw_n{n_days}.dat'),
            os.path.join(IMG_DIR, f'meta_n{n_days}.parquet'))

def build_store(df, n_days):
    dat, meta = store_paths(n_days)
    H, W = img_shape(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        log(f"N={n_days}: store already present, skipping generation."); return
    t0 = time.time()
    log(f"N={n_days}: generating images (gate_signals={SIGNAL_GATE})...")

    args = [(s, g.copy(), n_days, SIGNAL_GATE) for s, g in df.groupby('Symbol')]
    grouped, recs = {}, []
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        for sk, sdf, r in tqdm(ex.map(_index_symbol, args), total=len(args),
                               desc=f"index N={n_days}", leave=False):
            grouped[sk] = sdf; recs.extend(r)
    N = len(recs)
    if N == 0:
        log(f"N={n_days}: no valid windows."); return
    log(f"N={n_days}: {N:,} windows -> {N*H*W/1e9:.1f} GB raw ({H}x{W} px). rendering on {CPU} cores...")

    np.memmap(dat, dtype=np.uint8, mode='w+', shape=(N,1,H,W)).flush()
    m = pd.DataFrame(recs); m.attrs = {}
    m.to_parquet(meta)

    _G.update(dat=dat, shape=(N,1,H,W), n_days=n_days, grouped=grouped, recs=recs)
    chunk  = max(1, N // (CPU*8))
    ranges = [(i, min(i+chunk, N)) for i in range(0, N, chunk)]
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        list(tqdm(ex.map(_render_chunk, ranges), total=len(ranges),
                  desc=f"render N={n_days}", leave=False))
    _G.clear(); del grouped, recs, m
    free_memory()
    log(f"N={n_days}: built in {(time.time()-t0)/60:.1f} min -> {dat} | mem={mem_str()}")

def load_store(n_days, df=None):
    dat, meta = store_paths(n_days)
    if not (os.path.exists(dat) and os.path.exists(meta)):
        return None
    m = pd.read_parquet(meta).reset_index(drop=True)
    H, W = img_shape(n_days)
    N = len(m)
    expect, actual = N * H * W, os.path.getsize(dat)
    if actual != expect:
        raise RuntimeError(f"N={n_days}: .dat is {actual:,} bytes but metadata implies {expect:,} "
                           f"({N:,} imgs of {H}x{W}). Wrong dataset entirely.")
    m['_row'] = np.arange(N)                       # absolute row in the .dat -- fixed forever

    if DROP_INCOMPLETE:
        bad = m['end_idx'].values < (n_days - 1)
        if bad.any():
            log(f"N={n_days}: dropping {int(bad.sum()):,} blank rows (incomplete lookback).")
            m = m[~bad].reset_index(drop=True)

    if df is not None:
        ok, _ = align_store(m, df, verbose=False)
        n_bad = int((~ok).sum())
        if n_bad:
            log(f"N={n_days}: dropping {n_bad:,} rows ({100*(1-ok.mean()):.2f}%) whose images no "
                f"longer match the parquet (rendered before it changed). .dat untouched.")
            m = m[ok].reset_index(drop=True)
        if len(m) == 0:
            raise RuntimeError(f"N={n_days}: NO rows survive alignment. The parquet is not the one "
                               f"this store was rendered from. Re-render required.")

    log(f"N={n_days}: {N:,} images on disk ({actual/1e9:.1f} GB), {len(m):,} usable | mem={mem_str()}")
    return {'dat': dat, 'shape': (N, 1, H, W), 'meta': m}

## 4. Labels & splits — the exact-H-calendar-day fix

`end_idx + horizon` is a **positional** offset. On the equities calendar that meant "H trading days",
which was the intent. Here it would mean "H *bars*", and a bar is not guaranteed to be a day: an era
may contain gaps of up to 7 days, so a positional +5 could silently be a 12-calendar-day return.
That return would then be annualized by `365/5` — inflating it by up to 2.4x.

So the realized span is checked and **any sample whose forward window is not exactly `H` calendar
days is dropped**. The count is logged, not hidden. After this, `365/H` is the true periods-per-year.

The test set is **seed-independent and chronological**, so it is identical for every ensemble member
— that is what makes column-wise averaging of their predictions valid.

In [28]:
def label_for_horizon(store, df, horizon):
    m = store['meta']
    closes, dates = {}, {}
    for s, g in df.groupby('Symbol'):
        g = g.sort_values('Date')
        closes[s] = g['Close'].values.astype(np.float64)
        dates[s]  = g['Date'].values.astype('datetime64[D]')

    fwd = np.full(len(m), np.nan)
    n_span_bad = 0
    for s, pos in m.groupby('symbol').indices.items():
        c, dt = closes.get(s), dates.get(s)
        if c is None: continue
        ei  = m['end_idx'].values[pos]
        tgt = ei + horizon
        ok  = tgt < len(c)
        tgt_c = np.minimum(tgt, len(c) - 1)
        # ---- 24/7 fix: the holding period must be EXACTLY `horizon` CALENDAR days ----
        span = (dt[tgt_c] - dt[ei]).astype('timedelta64[D]').astype(int)
        exact = ok & (span == horizon)
        n_span_bad += int((ok & ~exact).sum())
        base_p, fwd_p = c[ei], c[tgt_c]
        with np.errstate(invalid='ignore', divide='ignore'):
            fwd[pos] = np.where(exact & (base_p != 0), fwd_p / base_p - 1.0, np.nan)
    del closes, dates

    d = m.copy(); d['Fwd_Ret'] = fwd
    n_before = len(d)
    d = d[~np.isnan(d['Fwd_Ret'].values)].reset_index(drop=True)
    log(f"  H={horizon}: {n_span_bad:,} samples dropped for a non-exact {horizon}-day span "
        f"(gap in the window); {n_before-len(d)-n_span_bad:,} dropped as undefined-label tail")
    rows  = d['_row'].values
    label = (d['Fwd_Ret'].values > 0).astype(np.int64)
    base  = d.rename(columns={'symbol': 'Symbol'})[['end_date','Symbol'] + META_COLS + ['Fwd_Ret']]
    base['Date'] = pd.to_datetime(base['end_date'])
    base = base.drop(columns='end_date')[['Date','Symbol'] + META_COLS + ['Fwd_Ret']]
    return rows, label, base

def trainval_positions(base):
    return np.where(base['Date'].values <= np.datetime64(TRAIN_END))[0]

def test_positions(base):
    d = base['Date'].values                       # seed-independent AND chronological
    te = np.where((d >= np.datetime64(OOS_START)) & (d <= np.datetime64(OOS_END)))[0]
    return te[np.argsort(d[te], kind='stable')]

def subsample_split(tv, seed, frac=TRAIN_FRAC):
    tv = tv.copy()
    np.random.default_rng(seed).shuffle(tv)
    cut = int(frac * len(tv))
    return tv[:cut], tv[cut:]

class ImgDataset(Dataset):
    def __init__(self, dat, shape, rows, labels=None):
        self.dat=dat; self.shape=tuple(int(x) for x in shape)
        self.rows=np.asarray(rows); self.labels=labels; self._mm=None
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if self._mm is None:                       # mapped lazily: one handle per worker
            self._mm = np.memmap(self.dat, dtype=np.uint8, mode='r', shape=self.shape)
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])]))
        if self.labels is None: return img
        return img, int(self.labels[i])
    def close(self):
        self._mm = None

## 5. CNN — verbatim (unchanged from the equities notebook / the paper)

In [29]:
def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, vs, vd = 2, 1, 1
        elif n_days == 20: blocks, vs, vd = 3, 3, 2
        else:              blocks, vs, vd = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (vs, 1) if i == 0 else (1, 1)
            dilation = (vd, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        H, W = img_shape(n_days)
        flat = self.features(torch.zeros(1,1,H,W)).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

## 6. Training engine — per-member checkpointing + explicit teardown

Every member writes `crypto_preds/pred_I{n}_R{h}_s{seed}.npy` on completion, so the run resumes
after a crash or a machine change. Models, loaders and worker processes are explicitly torn down
between members.

In [30]:
def _mk_loader(ds, shuffle, device):
    kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
              pin_memory=(device.startswith('cuda')))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(ds, **kw)

def _shutdown(loader):
    try:
        it = getattr(loader, '_iterator', None)
        if it is not None:
            it._shutdown_workers(); loader._iterator = None
    except Exception:
        pass

def pred_path(n_days, horizon, seed):
    return os.path.join(PRED_DIR, f'pred_I{n_days}_R{horizon}_s{seed}.npy')

def train_and_predict(job):
    n_days, horizon, dat, shape, tr, trl, va, val, te, device, seed, tag = job
    out = pred_path(n_days, horizon, seed)
    if os.path.exists(out):
        p = np.load(out)
        log(f"{tag} cached ({len(p):,} preds) -> skipping", tag='.')
        return p

    torch.manual_seed(seed); np.random.seed(seed)
    dev = torch.device(device)
    if dev.type == 'cuda': torch.cuda.set_device(dev)

    ds_tr = ImgDataset(dat, shape, tr, trl)
    ds_va = ImgDataset(dat, shape, va, val)
    ds_te = ImgDataset(dat, shape, te)
    l_tr = _mk_loader(ds_tr, True,  device)
    l_va = _mk_loader(ds_va, False, device)
    l_te = _mk_loader(ds_te, False, device)

    model     = TrendCNN(n_days).to(dev)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    amp    = USE_AMP and dev.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)

    log(f"{tag} start on {device} | train {len(tr):,} val {len(va):,} test {len(te):,} | {mem_str(dev)}")
    best, patience_ctr, t_start = float('inf'), 0, time.time()
    ckpt = tempfile.NamedTemporaryFile(suffix='.pt', delete=False).name

    for epoch in range(EPOCHS):
        t0 = time.time(); model.train(); tr_loss = 0.0
        for imgs, y in l_tr:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            y    = y.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if amp:
                with torch.autocast('cuda'):
                    loss = criterion(model(imgs), y)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(model(imgs), y)
                loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= max(1, len(l_tr))

        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for imgs, y in l_va:
                imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
                y    = y.to(dev, non_blocking=True)
                if amp:
                    with torch.autocast('cuda'):
                        va_loss += criterion(model(imgs), y).item()
                else:
                    va_loss += criterion(model(imgs), y).item()
        va_loss /= max(1, len(l_va))

        if va_loss < best:
            best, patience_ctr = va_loss, 0
            torch.save(model.state_dict(), ckpt); mark = 'best'
        else:
            patience_ctr += 1
            mark = f'patience {patience_ctr}/{PATIENCE}'
        log(f"{tag} ep{epoch+1:03d} train {tr_loss:.4f} val {va_loss:.4f} "
            f"{time.time()-t0:5.1f}s {mark} | {mem_str(dev)}")
        if patience_ctr >= PATIENCE:
            log(f"{tag} early stop at epoch {epoch+1} (best val {best:.4f})")
            break

    model.load_state_dict(torch.load(ckpt, map_location=dev, weights_only=True))
    os.remove(ckpt)

    model.eval(); probs = []
    with torch.no_grad():
        for imgs in l_te:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            if amp:
                with torch.autocast('cuda'):
                    p = torch.softmax(model(imgs), 1)[:, 1]
            else:
                p = torch.softmax(model(imgs), 1)[:, 1]
            probs.append(p.float().cpu().numpy())
    probs = np.concatenate(probs) if probs else np.array([])
    np.save(out, probs)

    for l in (l_tr, l_va, l_te): _shutdown(l)
    for d in (ds_tr, ds_va, ds_te): d.close()
    del l_tr, l_va, l_te, ds_tr, ds_va, ds_te, model, optimizer, criterion, scaler
    free_memory(dev)
    log(f"{tag} done in {(time.time()-t_start)/60:.1f} min | {len(probs):,} preds -> "
        f"{os.path.basename(out)} | {mem_str(dev)}")
    return probs

## 7. Portfolio evaluation — 365, the continuous rebalance grid, and the cost grid

**The rebalance calendar is a property of the strategy**, so it is derived from the full continuous
date grid and *not* from whatever dates survive a signal's warm-up. (Deriving it after `dropna`
would re-thin an already-spaced grid whenever a signal has a burn-in — e.g. TREND's — collapsing the
sample to a couple of periods.) Built with `pd.date_range(freq=f'{H}D')`, rebalances are **exactly H
calendar days apart**.

**TREND** is the genuine Han–Zhou–Zhu (2016) fitted expected-return forecast, not a moving-average
vote, and it is fitted **here** because it needs the horizon. `beta_t` needs a realized return so it
is only knowable at `t+H`; rebalance dates are spaced exactly `H` apart, so any beta from `s < t` is
realized by `t` — the forecast uses only already-realized information.

**Costs.** `cost_bps` is charged on **each leg per rebalance** (`2 x bps`), matching the equities
harness so the two studies stay comparable. State the assumption plainly: this bills one-way
turnover per leg. A full round trip (liquidate + re-enter, both legs) would be `4 x bps`, so **these
figures are a lower bound on costs** — read the 50 bps column as the ~25 bps round-trip case.

In [31]:
def rebalance_grid(dates, horizon):
    '''Rebalance dates EXACTLY `horizon` calendar days apart on the continuous 24/7 grid.'''
    d0, d1 = pd.Timestamp(np.min(dates)), pd.Timestamp(np.max(dates))
    return pd.date_range(d0, d1, freq=f'{horizon}D')

def add_trend_factor(ev, horizon, n_beta=12, min_names=30):
    '''Han-Zhou-Zhu (2016) trend factor. Real-time: no look-ahead.'''
    ev = ev.copy(); ev['TREND'] = np.nan
    rebal = rebalance_grid(ev['Date'].values, horizon)

    betas = {}
    for t in rebal:                                    # beta_t needs r(t -> t+H): only KNOWN at t+H
        g = ev[ev['Date'] == t].dropna(subset=MA_COLS + ['Fwd_Ret'])
        if len(g) < min_names: continue
        X = np.column_stack([np.ones(len(g)), g[MA_COLS].values])
        b, *_ = np.linalg.lstsq(X, g['Fwd_Ret'].values, rcond=None)
        betas[t] = b

    known = []                                         # betas already realized before this date
    for t in rebal:
        if len(known) >= n_beta:                       # burn-in: first n_beta dates get no forecast
            bbar = np.mean(known[-n_beta:], axis=0)
            g  = ev.loc[ev['Date'] == t, MA_COLS]
            ok = g.notna().all(axis=1)
            if ok.any():
                X = np.column_stack([np.ones(int(ok.sum())), g[ok].values])
                ev.loc[g[ok].index, 'TREND'] = X @ bbar
        if t in betas:
            known.append(betas[t])                     # realized at t+H -> usable from the next date
    return ev

def _ls_series(df, sig, H, min_names=20, cost_bps=0.0):
    '''Equal-weight decile long-short (D10 - D1), non-overlapping, net of 2 x cost_bps.'''
    d = df.dropna(subset=[sig, 'Fwd_Ret'])
    rebal = rebalance_grid(df['Date'].values, H)        # from the FULL grid, not the dropna'd one
    d = d[d['Date'].isin(rebal)]
    out = {}
    for dt, g in d.groupby('Date'):
        if len(g) < min_names: continue                # too thin to define 10 deciles
        dec = pd.qcut(g[sig].rank(method='first'), 10, labels=False)
        out[dt] = (g.loc[dec == 9, 'Fwd_Ret'].mean()
                   - g.loc[dec == 0, 'Fwd_Ret'].mean() - 2 * cost_bps / 1e4)
    return pd.Series(out).sort_index()

def _sharpe(r, H):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1) == 0 else \
           r.mean() / r.std(ddof=1) * np.sqrt(ANNUALIZER / H)     # LOCKED 365

def _tstat(r):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1) == 0 else \
           r.mean() / (r.std(ddof=1) / np.sqrt(len(r)))

def portfolio_performance(signal, df, horizon, min_names=20, cost_bps=0.0):
    r = _ls_series(df, signal, horizon, min_names=min_names, cost_bps=cost_bps)
    r = r.dropna(); n = len(r)
    if n < 2:
        return dict(AvgHoldRet=(r.mean() if n else np.nan), tstat=np.nan,
                    AnnReturn=np.nan, AnnSharpe=np.nan, NPeriods=n)
    ppy = ANNUALIZER / horizon                                     # LOCKED 365
    mu, sd = r.mean(), r.std(ddof=1)
    return dict(AvgHoldRet=mu, tstat=_tstat(r), AnnReturn=mu * ppy,
                AnnSharpe=(mu / sd * np.sqrt(ppy) if sd > 0 else np.nan), NPeriods=n)

## 8. Run — the locked 4-config grid

Resumes automatically: configs already in `crypto_results.csv` are skipped, and within a config any
member with a cached `.npy` is skipped. Stop and restart at any point.

In [32]:
# ---- generate any missing image stores ----
for n_days in IMAGE_SIZES:
    dat, meta = store_paths(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        continue
    build_store(df_market, n_days)

In [33]:
# ---- resume: what's already finished? ----
done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(prev['Config'].unique())
    log(f"resuming: {len(done)} configs already in {RESULTS_CSV} -> {sorted(done)}")
    results = prev.to_dict('records')
else:
    results = []

for n_days in IMAGE_SIZES:
    store = load_store(n_days, df_market)
    if store is None:
        log(f"N={n_days}: no store found, skipping."); continue

    for horizon in HORIZONS:
        cfg = f"I{n_days}/R{horizon}"
        if cfg in done:
            log(f"{cfg}: already complete, skipping."); continue
        log("=" * 66)
        log(f"CONFIG {cfg}  (ensemble {ENSEMBLE_SIZE}, {MAX_PARALLEL} concurrent on {BACKEND})")
        log("=" * 66)

        rows, label, base = label_for_horizon(store, df_market, horizon)
        te     = test_positions(base)
        tv_all = trainval_positions(base)
        if len(tv_all) == 0 or len(te) == 0:
            log(f"{cfg}: empty train or test split, skipping."); continue
        te_rows   = rows[te]
        eval_base = base.iloc[te].reset_index(drop=True)   # row-for-row aligned with every member
        log(f"{cfg}: train/val pool {len(tv_all):,} | test {len(te):,} "
            f"| test dates {eval_base['Date'].min().date()} -> {eval_base['Date'].max().date()}")

        jobs = []
        for i in range(ENSEMBLE_SIZE):
            seed = 1000 + i
            tr, va = subsample_split(tv_all, seed if BAGGING else SHARED_SPLIT_SEED)
            jobs.append((n_days, horizon, store['dat'], store['shape'],
                         rows[tr], label[tr], rows[va], label[va], te_rows,
                         DEVICES[i % len(DEVICES)], seed, f"[{cfg} m{i+1}/{ENSEMBLE_SIZE}]"))

        preds = [None] * ENSEMBLE_SIZE
        t_cfg = time.time()
        if MAX_PARALLEL > 1:
            with cf.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
                fut = {ex.submit(train_and_predict, j): k for k, j in enumerate(jobs)}
                for f in cf.as_completed(fut):
                    preds[fut[f]] = f.result()      # raises here if a member failed -- don't hide it
        else:
            for k, j in enumerate(jobs):
                preds[k] = train_and_predict(j)

        lens = {len(p) for p in preds}
        if lens != {len(te)}:
            raise RuntimeError(f"{cfg}: prediction lengths {lens} != test rows {len(te)}. "
                               f"Stale .npy from a different run? Delete {PRED_DIR} for this config.")

        eval_base['CNN_Prob'] = np.mean(preds, axis=0)
        eval_base = add_trend_factor(eval_base, horizon)
        log(f"{cfg}: ensemble complete in {(time.time()-t_cfg)/60:.1f} min | scoring...")

        for strat in ['CNN_Prob','MOM','STR','WSTR','TREND']:
            perf = portfolio_performance(strat, eval_base, horizon)
            results.append({'Config': cfg, 'Strategy': strat, 'CostBps': 0, **perf})
            log(f"  {strat:9s} AnnRet {perf['AnnReturn']:+.4f}  Sharpe {perf['AnnSharpe']:+.3f}  "
                f"t {perf['tstat']:+.2f}  periods {perf['NPeriods']}")

        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
        del preds, eval_base, rows, label, base, te, tv_all, te_rows
        free_memory()
        log(f"{cfg}: saved -> {RESULTS_CSV} | mem={mem_str()}")

    del store
    free_memory()

df_results = pd.DataFrame(results)
log(f"ALL DONE in {(time.time()-_T0)/60:.1f} min | mem={mem_str()}")
display(df_results)

[00:00:02.9] resuming: 2 configs already in ./crypto_results.csv -> ['I5/R20', 'I5/R5']
[00:00:03.2] N=5: 309,276 images on disk (0.1 GB), 309,276 usable | mem=rss 0.8G mps 0.1G
[00:00:03.2] I5/R5: already complete, skipping.
[00:00:03.2] I5/R20: already complete, skipping.
[00:00:03.9] N=20: 304,518 images on disk (1.2 GB), 304,518 usable | mem=rss 1.0G mps 0.1G
[00:00:03.9] ==================================================================
[00:00:03.9] CONFIG I20/R5  (ensemble 5, 1 concurrent on mps)
[00:00:03.9] ==================================================================
[00:00:04.2]   H=5: 4 samples dropped for a non-exact 5-day span (gap in the window); 704 dropped as undefined-label tail
[00:00:04.2] I20/R5: train/val pool 106,198 | test 197,612 | test dates 2021-01-01 -> 2026-06-25
[00:00:04.3] . [I20/R5 m1/5] cached (197,612 preds) -> skipping
[00:00:04.3] . [I20/R5 m2/5] cached (197,612 preds) -> skipping
[00:00:04.3] . [I20/R5 m3/5] cached (197,612 preds) -> skipping
[

,Config,Strategy,CostBps,AvgHoldRet,tstat,AnnReturn,AnnSharpe,NPeriods
0,I5/R5,CNN_Prob,0,0.012000,3.452558,0.875979,1.473093,401
1,I5/R5,MOM,0,-0.001581,-0.378487,-0.115391,-0.161488,401
2,I5/R5,STR,0,-0.003492,-0.682843,-0.254925,-0.291347,401
3,I5/R5,WSTR,0,-0.008224,-1.792597,-0.600365,-0.764842,401
4,I5/R5,TREND,0,0.002174,0.549528,0.158674,0.238054,389
5,I5/R20,CNN_Prob,0,0.002774,0.252347,0.050627,0.107803,100
6,I5/R20,MOM,0,-0.006870,-0.414799,-0.125386,-0.177202,100
7,I5/R20,STR,0,0.028327,1.981044,0.516962,0.846302,100
8,I5/R20,WSTR,0,-0.027899,-1.585147,-0.509151,-0.677175,100
9,I5/R20,TREND,0,0.013917,0.953805,0.253988,0.434360,88


## 9. Raw results (uncorrected — never report these alone)

In [34]:
df_results = pd.read_csv(RESULTS_CSV)
piv = df_results[df_results.CostBps == 0].pivot(index='Config', columns='Strategy', values='AnnSharpe')
order = [f'I{i}/R{h}' for i in IMAGE_SIZES for h in HORIZONS if f'I{i}/R{h}' in piv.index]
print(f"Annualized Sharpe (equal-weight decile long-short, non-overlapping, ann={ANNUALIZER:.0f}):")
display(piv.loc[order][[c for c in ['CNN_Prob','MOM','STR','WSTR','TREND'] if c in piv.columns]].round(3))
print("\nThese are RAW and UNCORRECTED. They are not a result until the next cell has run.")

Annualized Sharpe (equal-weight decile long-short, non-overlapping, ann=365):


Strategy,CNN_Prob,MOM,STR,WSTR,TREND
Config,,,,,
I5/R5,1.473,-0.161,-0.291,-0.765,0.238
I5/R20,0.108,-0.177,0.846,-0.677,0.434
I20/R5,0.783,-0.161,-0.291,-0.666,0.238
I20/R20,1.086,-0.177,0.846,-0.573,0.434



These are RAW and UNCORRECTED. They are not a result until the next cell has run.


## 10. Scientific analysis — the corrected verdict

Three things, in the order they must be read:

**Stage B (primary).** The family of tests **is** the 4-config grid. Bonferroni multiplier = 4;
BH at FDR 5%. If nothing survives, that is the honest headline and it is a real finding.

**Stage A (descriptive).** The strongest config is *characterized* — costs, stability, monotonicity,
block-bootstrap CI — and **never re-tested for significance on the same data** that selected it.
The block-bootstrap CI is the honest interval: it honours autocorrelation, which the t-stat does not.

**The survivorship decomposition — and here crypto can do what the equities study could not.**
The S&P 500 project could only *speculate* about the direction of its bias, because the dead names
were unavailable. Here **436 of the 618 universe assets are delisted and their full history is
present**, so the bias can be *measured*: score the true point-in-time universe, then deliberately
re-introduce the bias by restricting to coins still trading at the end, and take the difference.
**That difference is the survivorship bias, in Sharpe units.** It is the most interesting number
this study can produce, and it is worth more than the CNN result itself.

In [35]:
from math import erf, sqrt

def _p_two_sided(t):
    return np.nan if np.isnan(t) else 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2))))

def _block_boot_ci(r, H, n_boot=2000, seed=0):
    '''Stationary block bootstrap: honours autocorrelation, which the t-stat does not.'''
    r = r.dropna().values; n = len(r)
    if n < 10: return (np.nan, np.nan)
    avg_block = max(2, int(n ** (1/3))); p = 1.0 / avg_block
    rng = np.random.default_rng(seed); ann = np.sqrt(ANNUALIZER / H); est = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.empty(n, dtype=int); i = 0
        while i < n:
            start = rng.integers(0, n); L = rng.geometric(p)
            for k in range(L):
                if i >= n: break
                idx[i] = (start + k) % n; i += 1
        s = r[idx]
        est[b] = s.mean() / s.std(ddof=1) * ann if s.std(ddof=1) > 0 else 0.0
    return (np.percentile(est, 2.5), np.percentile(est, 97.5))

def _bh_reject(pvals, alpha=0.05):
    p = np.array([x if not np.isnan(x) else 1.0 for x in pvals]); m = len(p)
    order = np.argsort(p); passed = p[order] <= (np.arange(1, m+1) / m) * alpha
    rej = np.zeros(m, bool)
    if passed.any(): rej[order[:np.where(passed)[0].max()+1]] = True
    return rej

def _rebuild(n_days, horizon):
    '''Rebuild eval_base from CACHED predictions -- no retraining.'''
    store = load_store(n_days, df_market)
    if store is None: return None
    rows, label, base = label_for_horizon(store, df_market, horizon)
    te = test_positions(base); eb = base.iloc[te].reset_index(drop=True)
    ps = [np.load(pred_path(n_days, horizon, 1000+i)) for i in range(ENSEMBLE_SIZE)
          if os.path.exists(pred_path(n_days, horizon, 1000+i))]
    del store; free_memory()
    if not ps or {len(p) for p in ps} != {len(te)}: return None
    eb['CNN_Prob'] = np.mean(ps, axis=0)
    return add_trend_factor(eb, horizon)

CONFIGS = [(n, h) for n in IMAGE_SIZES for h in HORIZONS
           if os.path.exists(pred_path(n, h, 1000))]
log(f"configs with cached predictions: {[f'I{n}/R{h}' for n,h in CONFIGS]}")
EB = {}
for n, h in CONFIGS:
    eb = _rebuild(n, h)
    if eb is not None: EB[(n, h)] = eb

# ============================================================================
print("\n" + "="*72)
print("STAGE B  —  FULL-GRID MULTIPLE-TESTING CORRECTION  (the primary result)")
print("="*72)
print(f"The family of tests IS the locked {N_CONFIGS}-config grid. Report the CORRECTED verdict.\n")

grid = []
for (n, h), eb in EB.items():
    r = _ls_series(eb, 'CNN_Prob', h); t = _tstat(r)
    grid.append({'Config': f'I{n}/R{h}', 'Sharpe': _sharpe(r, h), 'tstat': t,
                 'raw_p': _p_two_sided(t), 'N': len(r.dropna())})
gdf = pd.DataFrame(grid)
if len(gdf):
    gdf['bonferroni_p'] = np.minimum(gdf['raw_p'] * N_CONFIGS, 1.0)   # LOCKED multiplier = 4
    gdf['BH_reject']    = _bh_reject(gdf['raw_p'].values)
    gdf = gdf.sort_values('raw_p').reset_index(drop=True)
    print(gdf.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    n_survive = int(gdf['BH_reject'].sum())
    print(f"\n  tests in family        : {N_CONFIGS}")
    print(f"  survive BH (FDR 5%)    : {n_survive}")
    print(f"  survive Bonferroni     : {int((gdf['bonferroni_p'] < 0.05).sum())}")
    print("\n  PRIMARY VERDICT: " + (
          f"{n_survive} config(s) survive correction -- a genuine, if modest, effect."
          if n_survive else
          "NO config survives correction across the searched family. The apparent\n"
          "  per-config signals are consistent with multiple-testing noise. This is the\n"
          "  honest headline, and it is a real finding, not a failure."))

# ============================================================================
if len(gdf):
    print("\n" + "="*72)
    print("STAGE A  —  CHARACTERIZE THE STRONGEST CONFIG  (descriptive, NOT a p-value)")
    print("="*72)
    best = gdf.iloc[0]['Config']
    bn, bh_ = int(best[1:best.index('/')]), int(best[best.index('R')+1:])
    print(f"strongest raw config: {best}  (selected post hoc -> characterize, don't re-test)\n")
    eb = EB[(bn, bh_)]
    r  = _ls_series(eb, 'CNN_Prob', bh_)
    lo, hi = _block_boot_ci(r, bh_)
    print("[A0] Block-bootstrap Sharpe CI (honours autocorrelation, the t-stat does not)")
    print(f"     Sharpe {_sharpe(r,bh_):+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]   "
          f"-> {'excludes 0' if (lo > 0 or hi < 0) else 'STRADDLES 0 (weak)'}\n")

    print(f"[A1] Transaction costs (LOCKED grid, bps per side; 2x charged per rebalance)")
    for bps in COST_GRID:
        rc = _ls_series(eb, 'CNN_Prob', bh_, cost_bps=bps)
        print(f"     {bps:2d} bps/side : Sharpe {_sharpe(rc,bh_):+.3f}  AnnRet {rc.mean()*ANNUALIZER/bh_:+.4f}")
    print()

    print("[A2] Rolling 3-year Sharpe")
    idx = pd.to_datetime(r.index); shs = []
    for y0 in sorted(set(idx.year)):
        sub = r[(idx.year >= y0) & (idx.year < y0+3)]
        if len(sub) > 20: shs.append(_sharpe(sub, bh_))
    if shs:
        print(f"     windows: {len(shs)} | positive: {np.mean([s>0 for s in shs]):.0%} | "
              f"min {min(shs):+.2f} max {max(shs):+.2f}\n")

    print("[A3] Decile monotonicity")
    d = eb.dropna(subset=['CNN_Prob','Fwd_Ret'])
    d = d[d['Date'].isin(rebalance_grid(eb['Date'].values, bh_))]
    means = np.zeros(10); c = 0
    for _, g in d.groupby('Date'):
        if len(g) < 20: continue
        dec = pd.qcut(g['CNN_Prob'].rank(method='first'), 10, labels=False)
        for k in range(10): means[k] += g.loc[dec == k, 'Fwd_Ret'].mean()
        c += 1
    if c:
        means /= c; rho = np.corrcoef(np.arange(10), means)[0,1]
        print(f"     decile means (bps): " + " ".join(f"{m*1e4:+.0f}" for m in means))
        print(f"     rank corr decile->return: {rho:+.2f}  (>0.5 = coherent; flat = tail-only)\n")

    print("[A4] Long vs short leg")
    longs, shorts = [], []
    for _, g in d.groupby('Date'):
        if len(g) < 20: continue
        dec = pd.qcut(g['CNN_Prob'].rank(method='first'), 10, labels=False); mk = g['Fwd_Ret'].mean()
        longs.append(g.loc[dec==9,'Fwd_Ret'].mean() - mk)
        shorts.append(mk - g.loc[dec==0,'Fwd_Ret'].mean())
    if longs:
        print(f"     long  (D9-mkt): Sharpe {_sharpe(pd.Series(longs), bh_):+.3f}")
        print(f"     short (mkt-D0): Sharpe {_sharpe(pd.Series(shorts), bh_):+.3f}")

# ============================================================================
print("\n" + "="*72)
print("HEADLINE  —  SURVIVORSHIP DECOMPOSITION  (crypto can MEASURE what equities could only guess)")
print("="*72)
# 'Dead' is derived from the panel itself: an asset whose last bar predates the end of the
# sample by >30d was delisted. No schema change needed, and it matches exchangeInfo's BREAK set.
last_seen = df_market.groupby('Symbol')['Date'].max()
alive_cut = df_market['Date'].max() - pd.Timedelta(days=30)
alive     = set(last_seen[last_seen >= alive_cut].index)
n_dead    = int((~last_seen.index.isin(alive)).sum())
print(f"assets in panel: {len(last_seen):,} | still trading at the end: {len(alive):,} | "
      f"delisted: {n_dead:,}\n")

rows_sv = []
for (n, h), eb in EB.items():
    full = _sharpe(_ls_series(eb, 'CNN_Prob', h), h)                      # true point-in-time
    surv = _sharpe(_ls_series(eb[eb['Symbol'].isin(alive)], 'CNN_Prob', h), h)  # bias re-introduced
    rows_sv.append({'Config': f'I{n}/R{h}', 'Sharpe_true_PIT': full,
                    'Sharpe_survivors_only': surv, 'bias': surv - full})
    print(f"  I{n}/R{h}: true point-in-time {full:+.3f} | survivors-only {surv:+.3f} "
          f"| BIAS {surv-full:+.3f}")
sv = pd.DataFrame(rows_sv)
if len(sv):
    print(f"\n  mean survivorship bias across the grid: {sv['bias'].mean():+.3f} Sharpe")
    print("  A POSITIVE bias means dropping the dead coins FLATTERS the strategy -- i.e. a")
    print("  survivor-selected crypto universe manufactures Sharpe out of nothing. That is the")
    print("  number the equities study could never measure, and it is reported here regardless")
    print("  of whether the CNN itself works.")

# ---- OOS sub-period stability ----
print("\n" + "-"*72)
print("OOS SUB-PERIOD STABILITY")
print("-"*72)
splits = {'early 2021-2023': ('2021-01-01','2023-12-31'),
          'late  2024-2026': ('2024-01-01','2026-06-30')}
for (n, h), eb in EB.items():
    parts = {k: _sharpe(_ls_series(eb[(eb.Date >= a) & (eb.Date <= b)], 'CNN_Prob', h), h)
             for k, (a, b) in splits.items()}
    full = _sharpe(_ls_series(eb, 'CNN_Prob', h), h)
    print(f"  I{n}/R{h}: full {full:+.2f} | " +
          " | ".join(f"{k} {v:+.2f}" for k, v in parts.items()))

if len(gdf):
    gdf.to_csv(os.path.join(OUT_DIR, 'crypto_corrected_grid.csv'), index=False)
    sv.to_csv(os.path.join(OUT_DIR, 'crypto_survivorship_decomp.csv'), index=False)
    print(f"\nsaved -> crypto_corrected_grid.csv, crypto_survivorship_decomp.csv")
print("\nNOTE: no benchmark comparison yet -- that is Stage 3. A CNN Sharpe without")
print("      buy-and-hold BTC and momentum beside it is not a result. Do not quote it.")

[00:00:10.4] configs with cached predictions: ['I5/R5', 'I5/R20', 'I20/R5', 'I20/R20']
[00:00:10.7] N=5: 309,276 images on disk (0.1 GB), 309,276 usable | mem=rss 0.6G mps 0.1G
[00:00:10.8]   H=5: 4 samples dropped for a non-exact 5-day span (gap in the window); 704 dropped as undefined-label tail
[00:00:11.7] N=5: 309,276 images on disk (0.1 GB), 309,276 usable | mem=rss 0.8G mps 0.1G
[00:00:11.8]   H=20: 4 samples dropped for a non-exact 20-day span (gap in the window); 2,732 dropped as undefined-label tail
[00:00:12.1] N=20: 304,518 images on disk (1.2 GB), 304,518 usable | mem=rss 0.8G mps 0.1G
[00:00:12.2]   H=5: 4 samples dropped for a non-exact 5-day span (gap in the window); 704 dropped as undefined-label tail
[00:00:13.0] N=20: 304,518 images on disk (1.2 GB), 304,518 usable | mem=rss 0.8G mps 0.1G
[00:00:13.1]   H=20: 4 samples dropped for a non-exact 20-day span (gap in the window); 2,732 dropped as undefined-label tail

STAGE B  —  FULL-GRID MULTIPLE-TESTING CORRECTION  (th